# FASE 2 · LIMPIEZA Y TRANSFORMACIÓN DE LOS DATOS

A partir del análisis exploratorio realizado en la fase anterior, se han identificado inconsistencias, valores nulos y variables que requieren transformación para poder realizar un análisis más preciso y fiable.

En esta fase se lleva a cabo la limpieza y transformación del dataset, con el objetivo de obtener una base de datos estructurada, coherente y lista para su posterior análisis. Para ello, se abordan aspectos como el tratamiento de valores nulos, la eliminación de variables irrelevantes, la homogeneización de formatos y la creación o ajuste de variables cuando es necesario.

El resultado de esta fase será un dataset limpio y preparado, que permitirá analizar de forma más rigurosa los factores relacionados con el attrition en la siguiente fase del proyecto.

In [2]:
# Importación de librerías

import pandas as pd
import re

# Añadimos la carpeta /src al path para poder importar funciones propias desde los notebooks

from pathlib import Path
import sys

sys.path.append(str(Path("../src").resolve()))

from soporte_nulos import imputar_moda, imputar_mediana

# Configuración para visualizar todas las columnas del DataFrame
pd.set_option("display.max_columns", None)

In [ ]:
# Carga del dataset

df_hr = pd.read_csv("../files/raw/hr.csv")

# # Creamos una copia de trabajo para preservar el dataset original

df_hr_copia = df_hr.copy()

In [4]:
# Eliminación de columnas irrelevantes o redundantes

df_hr_copia.drop(["EmployeeCount", "StandardHours", "Over18"], axis=1, inplace=True)

df_hr_copia.drop(["DailyRate", "HourlyRate", "MonthlyRate"], axis=1, inplace=True)

df_hr_copia.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   Department                1445 non-null   str    
 4   DistanceFromHome          1474 non-null   int64  
 5   Education                 1474 non-null   int64  
 6   EducationField            1416 non-null   str    
 7   EmployeeNumber            1474 non-null   int64  
 8   EnvironmentSatisfaction   1474 non-null   int64  
 9   Gender                    1474 non-null   str    
 10  JobInvolvement            1474 non-null   int64  
 11  JobLevel                  1474 non-null   int64  
 12  JobRole                   1474 non-null   str    
 13  JobSatisfaction           1445 non-null   float64
 14  MaritalStatus      

In [5]:
# Eliminación de registros duplicados

df_hr_copia.drop_duplicates(inplace=True)

print(f"Dimensiones tras eliminar duplicados: {df_hr_copia.shape}")

Dimensiones tras eliminar duplicados: (1470, 29)


## Estandarización y corrección de variables categóricas

In [6]:
# Eliminamos espacios innecesarios y unificamos el estilo de escritura en "JobRole"

df_hr_copia["JobRole"] = df_hr_copia["JobRole"].str.strip().str.title()

In [7]:
# Corrección de error ortográfico detectado en "MaritalStatus"

df_hr_copia['MaritalStatus'] = df_hr_copia['MaritalStatus'].replace("Marreid", "Married")

# Revisión de que los cambios se hayan aplicado
df_hr_copia['MaritalStatus'].unique()

<StringArray>
['Single', 'Married', 'Divorced', nan]
Length: 4, dtype: str

## Revisión inicial de valores nulos

In [8]:
# Porcentaje de valores nulos por columna

porcentaje_nulos = round((df_hr_copia.isnull().sum() / df_hr_copia.shape[0] * 100), 2)

porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)

YearsWithCurrManager     10.00
MaritalStatus             8.98
BusinessTravel            7.96
TrainingTimesLastYear     5.99
Age                       4.97
EducationField            3.95
OverTime                  2.99
Department                1.97
JobSatisfaction           1.97
MonthlyIncome             0.95
dtype: float64

In [9]:
# Comprobamos también el número de valores nulos

df_hr_copia.isnull().sum()

Age                          73
Attrition                     0
BusinessTravel              117
Department                   29
DistanceFromHome              0
Education                     0
EducationField               58
EmployeeNumber                0
EnvironmentSatisfaction       0
Gender                        0
JobInvolvement                0
JobLevel                      0
JobRole                       0
JobSatisfaction              29
MaritalStatus               132
MonthlyIncome                14
NumCompaniesWorked            0
OverTime                     44
PercentSalaryHike             0
PerformanceRating             0
RelationshipSatisfaction      0
StockOptionLevel              0
TotalWorkingYears             0
TrainingTimesLastYear        88
WorkLifeBalance               0
YearsAtCompany                0
YearsInCurrentRole            0
YearsSinceLastPromotion       0
YearsWithCurrManager        147
dtype: int64

## Imputación general mediante funciones reutilizables

Aplicación de funciones de soporte para aquellas variables cuya imputación puede resolverse de forma homogénea:

- **Moda** para variables categóricas o discretas donde el valor más frecuente es una aproximación razonable.
- **Mediana por grupo** para variables numéricas donde conviene mantener la lógica interna del dato y reducir el efecto de posibles valores extremos.

In [10]:
# Variables imputadas con la moda
lista_moda = ["MaritalStatus", "BusinessTravel", "TrainingTimesLastYear"]

# Variables imputadas con la mediana dentro de un grupo
dicc_mediana = {"YearsWithCurrManager": "JobLevel", "Age": "JobLevel"}

df_hr_copia = imputar_moda(df_hr_copia, lista_moda)
df_hr_copia = imputar_mediana(df_hr_copia, dicc_mediana)

Columna 'MaritalStatus': valores nulos reemplazados por la moda -> 'Married'
Columna 'BusinessTravel': valores nulos reemplazados por la moda -> 'Travel_Rarely'
Columna 'TrainingTimesLastYear': valores nulos reemplazados por la moda -> '2.0'
Columna 'YearsWithCurrManager': valores nulos imputados por la mediana de 'JobLevel'
Columna 'Age': valores nulos imputados por la mediana de 'JobLevel'


## Imputaciones específicas según la naturaleza de cada variable

In [11]:
# "EducationField"

# Eliminación de espacios y estandarización del formato
df_hr_copia["EducationField"] = df_hr_copia["EducationField"].str.strip().str.title()

# Imputación según la moda dentro de cada Department
df_hr_copia["EducationField"] = df_hr_copia.groupby("Department")["EducationField"].transform(lambda x: x.fillna(x.mode()[0]))

# Tras observar que siguen quedando nulos, realizamos una segunda imputación con la moda
df_hr_copia["EducationField"] = df_hr_copia["EducationField"].fillna(df_hr_copia["EducationField"].mode()[0])


In [12]:
# "OverTime" - Uso de "Unknown" para no imputar un comportamiento no observable

df_hr_copia["OverTime"] = df_hr_copia["OverTime"].fillna("Unknown")

In [13]:
# "Department" - Imputación según la moda de JobRole. Ambas variables guardan relación lógica

df_hr_copia["Department"] = df_hr_copia.groupby("JobRole")["Department"].transform(lambda x: x.fillna(x.mode()[0]))

In [14]:
# "JobRole" - Cambio del puesto "Manager" a sus categorias por departamento

df_hr_copia.loc[df_hr_copia['JobRole'] == 'Manager','JobRole'] = (df_hr_copia['Department'] + ' Manager')

In [15]:
# "JobSatisfaction" - Al tratarse de una variable discreta, imputamos con la moda

df_hr_copia["JobSatisfaction"] = df_hr_copia["JobSatisfaction"].fillna(df_hr_copia["JobSatisfaction"].mode()[0])

In [16]:
# "MonthlyIncome" - Usamos la mediana por JobRole para mantener coherencia interna y reducir el impacto de posibles valores extremos en la distribución salarial

df_hr_copia["MonthlyIncome"] = df_hr_copia.groupby("JobRole")["MonthlyIncome"].transform(lambda x: x.fillna(x.median()))

## Validación final de la imputación

In [17]:
# Comprobamos el estado final de los valores nulos

porcentaje_nulos_final = round((df_hr_copia.isnull().sum() / df_hr_copia.shape[0]) * 100, 2)
porcentaje_nulos_final[porcentaje_nulos_final > 0].sort_values(ascending=False)

Series([], dtype: float64)

In [18]:
# Conversión de float a int de variables discretas

columnas = ["Age", "TrainingTimesLastYear", "YearsWithCurrManager", "JobSatisfaction"]
df_hr_copia[columnas]= df_hr_copia[columnas].astype(int)


In [19]:
# Homogeneizamos nombres de columnas y categorías

# Convertimos los nombres de columnas a formato Title_Case (con "_")
df_hr_copia.columns = [re.sub(r'(?<!^)(?=[A-Z])', "_", col) for col in df_hr_copia.columns]

# Sustituimos "_" por espacios en BusinessTravel para mejorar legibilidad
df_hr_copia["Business_Travel"] = df_hr_copia["Business_Travel"].str.replace("_", " ", regex=False)

In [20]:
# Verificación final del dataset limpio

print("Duplicados:", df_hr_copia.duplicated().sum())
print("Nulos totales:", df_hr_copia.isnull().sum().sum())

Duplicados: 0
Nulos totales: 0


In [ ]:
# Tras aplicar las transformaciones, exportamos nuestro dataset en un nuevo fichero

df_hr_copia.to_csv("../files/processed/datos_limpios.csv", index=False)

### Conclusión

En esta fase se ha llevado a cabo la limpieza y transformación del dataset, resolviendo los problemas identificados en el análisis exploratorio.

Se han tratado los valores nulos mediante distintos métodos de imputación, se han eliminado variables irrelevantes y se han homogeneizado formatos y estructuras para garantizar la coherencia de los datos. Como resultado, se obtiene un dataset limpio, consistente y preparado para el análisis.

Esta base de datos permite avanzar hacia la fase de visualización y análisis, donde se estudiarán en detalle los factores que influyen en el attrition.